# Clase 6 · Notebook 01
# La convolución y el pooling, desde cero

**Introducción al Deep Learning — Módulo 3, Unidad 1: Redes de neuronas convolucionales**

Antes de usar Keras, vamos a **ver con nuestros ojos** qué hace una capa convolucional y una de pooling.
Aplicaremos filtros (kernels) sobre una imagen y observaremos el **mapa de características** resultante.

1. Una imagen es una matriz de píxeles.
2. La convolución: aplicar un filtro para resaltar bordes.
3. El pooling: reducir el tamaño conservando lo importante.

> 💡 Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Una imagen es una matriz

Cargamos una imagen en escala de grises del dataset `digits` de Scikit-Learn (dígitos manuscritos 8×8)
y la mostramos junto con sus valores numéricos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[0]          # un "0" de 8x8 píxeles
print("Forma de la imagen:", img.shape, "(filas x columnas)")

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("Imagen (8x8)")
ax[1].imshow(img, cmap="gray")
for (i, j), v in np.ndenumerate(img):
    ax[1].text(j, i, int(v), ha="center", va="center", color="#FF647E", fontsize=8)
ax[1].set_title("Valores de los píxeles")
plt.tight_layout(); plt.show()

## 2. La convolución: aplicar un filtro

Una **convolución** desliza un filtro (kernel) por la imagen y, en cada posición, multiplica y suma.
Resalta cierta información. Definimos la operación a mano para entenderla.

In [ ]:
def convolucion(imagen, filtro):
    fh, fw = filtro.shape
    ih, iw = imagen.shape
    salida = np.zeros((ih - fh + 1, iw - fw + 1))
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            region = imagen[i:i+fh, j:j+fw]      # sección de la imagen
            salida[i, j] = np.sum(region * filtro)  # multiplicar y sumar
    return salida

# Filtros clásicos de detección de bordes (Sobel)
borde_vertical = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]])
borde_horizontal = borde_vertical.T
print("Filtro de bordes verticales:\n", borde_vertical)

Usaremos una imagen un poco más grande (un dígito reescalado) para que los bordes se vean mejor.

In [ ]:
from scipy.ndimage import zoom
grande = zoom(img, 6, order=1)   # ampliamos la imagen 8x8 -> 48x48

mapa_v = convolucion(grande, borde_vertical)
mapa_h = convolucion(grande, borde_horizontal)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(grande, cmap="gray");  ax[0].set_title("Original")
ax[1].imshow(np.abs(mapa_v), cmap="gray"); ax[1].set_title("Filtro: bordes verticales")
ax[2].imshow(np.abs(mapa_h), cmap="gray"); ax[2].set_title("Filtro: bordes horizontales")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

Cada filtro resalta una característica distinta (bordes en una dirección). En una CNN, **la red aprende
los valores de estos filtros** automáticamente durante el entrenamiento. El resultado es el **mapa de
características**.

## 3. El pooling: reducir el tamaño

El **max-pooling** divide la imagen en regiones y se queda con el **valor máximo** de cada una.
Reduce el tamaño (y el cómputo) conservando la información más fuerte.

In [ ]:
def max_pooling(imagen, tam=2):
    ih, iw = imagen.shape
    oh, ow = ih // tam, iw // tam
    salida = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = imagen[i*tam:(i+1)*tam, j*tam:(j+1)*tam]
            salida[i, j] = region.max()
    return salida

reducido = max_pooling(np.abs(mapa_v), tam=2)
print("Antes del pooling:", np.abs(mapa_v).shape)
print("Después del pooling 2x2:", reducido.shape)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(np.abs(mapa_v), cmap="gray"); ax[0].set_title("Mapa de características")
ax[1].imshow(reducido, cmap="gray");       ax[1].set_title("Tras max-pooling 2x2")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

La imagen se ha reducido a la mitad en cada dimensión, pero conserva los bordes principales.

## Resumen

- Una imagen es una **matriz de píxeles** (valores numéricos).
- La **convolución** aplica un filtro para resaltar características (bordes, texturas…) → **mapa de características**.
- En una CNN, los filtros **se aprenden** durante el entrenamiento.
- El **max-pooling** reduce el tamaño conservando lo más relevante.

➡️ En el **Notebook 02** dejaremos que Keras aprenda los filtros: construiremos una CNN completa.